# Proyecto 2 - Extracción de Características

## 02_Feature_Engineering.ipynb

Este notebook crea nuevas variables a partir de los datos climáticos y prepara el dataset para modelos de clasificación y series de tiempo.

### Objetivos
- Extraer características temporales de la columna `date`
- Calcular indicadores como razón humedad/presión y promedios móviles
- Generar etiquetas de clasificación de temperatura
- Guardar un dataset preparado para modelado

In [1]:
import pandas as pd
import numpy as np

train_path = '../data/DailyDelhiClimateTrain.csv'
test_path = '../data/DailyDelhiClimateTest.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

## Creación de características temporales

In [2]:
for df in [train_df, test_df]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

train_df[['date', 'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend']].head()

,date,year,month,day,dayofweek,quarter,is_weekend
0,2013-01-01,2013,1,1,1,1,0
1,2013-01-02,2013,1,2,2,1,0
2,2013-01-03,2013,1,3,3,1,0
3,2013-01-04,2013,1,4,4,1,0
4,2013-01-05,2013,1,5,5,1,1


## Nuevas variables climáticas

In [3]:
for df in [train_df, test_df]:
    df['humidity_pressure_ratio'] = df['humidity'] / df['meanpressure']
    df['temp_diff'] = df['meantemp'].diff().fillna(0)
    df['pressure_diff'] = df['meanpressure'].diff().fillna(0)
    df['humidity_diff'] = df['humidity'].diff().fillna(0)

train_df[['humidity_pressure_ratio', 'temp_diff', 'pressure_diff', 'humidity_diff']].head()

,humidity_pressure_ratio,temp_diff,pressure_diff,humidity_diff
0,0.083197,0.000000,0.000000,0.000000
1,0.090391,-2.600000,2.133333,7.500000
2,0.085406,-0.233333,0.866667,-5.000000
3,0.070129,1.500000,-1.500000,-15.666667
4,0.085424,-2.666667,-0.666667,15.500000


## Promedios móviles y ventanas temporales

In [4]:
window_cols = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
for df in [train_df, test_df]:
    for col in window_cols:
        df[f'{col}_rolling_3'] = df[col].rolling(window=3, min_periods=1).mean()
        df[f'{col}_rolling_7'] = df[col].rolling(window=7, min_periods=1).mean()

train_df[[col for col in train_df.columns if 'rolling' in col]].head()

,meantemp_rolling_3,meantemp_rolling_7,humidity_rolling_3,humidity_rolling_7,wind_speed_rolling_3,wind_speed_rolling_7,meanpressure_rolling_3,meanpressure_rolling_7
0,10.000000,10.000000,84.500000,84.500000,0.000000,0.000000,1015.666667,1015.666667
1,8.700000,8.700000,88.250000,88.250000,1.490000,1.490000,1016.733333,1016.733333
2,8.188889,8.188889,87.833333,87.833333,2.537778,2.537778,1017.377778,1017.377778
3,7.744444,8.308333,83.444444,83.708333,2.948889,2.211667,1017.877778,1017.325000
4,7.277778,7.846667,81.722222,84.333333,3.188889,2.509333,1017.444444,1017.160000


## Etiquetas de clasificación de temperatura

Convertimos la variable `meantemp` en una categoría sencilla para entrenar modelos de clasificación.

In [5]:
labels = [
    (train_df['meantemp'] < 15),
    (train_df['meantemp'] >= 15) & (train_df['meantemp'] < 25),
    (train_df['meantemp'] >= 25)
]
choices = ['cold', 'mild', 'hot']
train_df['temp_category'] = np.select(labels, choices)
train_df['temp_category'].value_counts()

temp_category
hot     851
mild    451
cold    160
Name: count, dtype: int64

## Preparación final de columnas para modelado

In [6]:
feature_columns = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend',
    'humidity', 'wind_speed', 'meanpressure', 'humidity_pressure_ratio',
    'meantemp_rolling_3', 'meantemp_rolling_7',
    'humidity_rolling_3', 'humidity_rolling_7',
    'wind_speed_rolling_3', 'wind_speed_rolling_7',
    'meanpressure_rolling_3', 'meanpressure_rolling_7'
]

train_df[feature_columns].head()

,year,month,day,dayofweek,quarter,is_weekend,humidity,wind_speed,meanpressure,humidity_pressure_ratio,meantemp_rolling_3,meantemp_rolling_7,humidity_rolling_3,humidity_rolling_7,wind_speed_rolling_3,wind_speed_rolling_7,meanpressure_rolling_3,meanpressure_rolling_7
0,2013,1,1,1,1,0,84.500000,0.000000,1015.666667,0.083197,10.000000,10.000000,84.500000,84.500000,0.000000,0.000000,1015.666667,1015.666667
1,2013,1,2,2,1,0,92.000000,2.980000,1017.800000,0.090391,8.700000,8.700000,88.250000,88.250000,1.490000,1.490000,1016.733333,1016.733333
2,2013,1,3,3,1,0,87.000000,4.633333,1018.666667,0.085406,8.188889,8.188889,87.833333,87.833333,2.537778,2.537778,1017.377778,1017.377778
3,2013,1,4,4,1,0,71.333333,1.233333,1017.166667,0.070129,7.744444,8.308333,83.444444,83.708333,2.948889,2.211667,1017.877778,1017.325000
4,2013,1,5,5,1,1,86.833333,3.700000,1016.500000,0.085424,7.277778,7.846667,81.722222,84.333333,3.188889,2.509333,1017.444444,1017.160000


## Guardar dataset preparado

Guardamos los datos listos para modelado en el directorio `notebooks`.

In [7]:
train_df.to_csv('../data/train_feature_engineered.csv', index=False)
test_df.to_csv('../data/test_feature_engineered.csv', index=False)
print('Archivos guardados: train_feature_engineered.csv, test_feature_engineered.csv')

Archivos guardados: train_feature_engineered.csv, test_feature_engineered.csv
